# 01 — SkillRet labels → training examples

This notebook demonstrates the conversion using **synthetic fixtures only**. The real dataset is downloaded later by a pinned script. A qrel is one positive query–skill link; it is not a separate independent query.

In [ ]:
from pathlib import Path
import sys, json
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'skillret_decider').is_dir())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
print('Project:', ROOT.name)
print('No paid resources or real models are started by this notebook.')

In [ ]:
from skillret_decider.core import read_jsonl, query_metrics
examples = read_jsonl(ROOT / 'examples/synthetic/converted.synthetic.jsonl')
for e in examples:
    print('\nQuery:', e['state'])
    for key, description in e['question']['criteria'].items():
        print(key, e['gold_probs'][key], description)

## Multiple positives and NONE

Two relevant candidates receive target mass 0.5 each. When the retriever misses the relevant skill, NONE receives 1.0. These are categorical loss targets, **not independently calibrated probabilities**. Never inject the correct skill into evaluation candidates.

In [ ]:
print(json.dumps(examples[0]['question'], indent=2))
print('Soft target:', examples[0]['gold_probs'])
assert all(abs(sum(e['gold_probs'].values()) - 1) < 1e-9 for e in examples)

## Recall is not Hit

If there are two labelled relevant skills and the first result is one of them, Hit@1 is 1 but Recall@1 is 0.5. The full gold set stays in the denominator even if retrieval missed a member.

In [ ]:
metrics = query_metrics(['review', 'commit', 'status'], {'review', 'security'})
print({k: metrics[k] for k in ['hit@1', 'recall@1', 'recall@5', 'mrr@15']})

## Actual data preparation

After the RunPod budget is approved, run these commands from the project root. They are text here, not automatically executed.

```bash
python -m scripts.fetch --out workspace/raw/train
python -m scripts.build --raw workspace/raw/train --config configs/pilot.json --out workspace/data-pilot
```

Use `--prior-split verified_split.json` only with the original CLM split IDs. Counts alone cannot recreate a split. Read PROTOCOL.md before changing the representation or candidate count.